# 🐍 01 · Python & NumPy for Bioimages
### Images are just arrays of numbers — once you believe that, everything else follows

**Part A · Foundations** · Module 01 of 37 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/01_python_numpy_for_images.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, Matplotlib |
| **Prerequisites** | Module 00 |
| **Estimated time** | 60–75 minutes |
| **02-740 topics covered** | Programming/NumPy proficiency — a prerequisite skill the course assumes and this module builds from scratch |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> The 'image' in this notebook is a tiny, hand-built synthetic array — no file is ever opened or downloaded.

## 🎯 Learning objectives

1. Explain why a digital image is, underneath everything, just a grid of numbers.
2. Create, index, slice, and crop NumPy arrays representing single- and multi-channel images.
3. Perform basic arithmetic on images (addition, scaling, clipping) and understand why data type and
   value range matter.
4. Visualize arrays with Matplotlib, including grids of multiple images side by side.
5. Use a seeded random number generator so your results are exactly reproducible.

## 🗺️ Contents

1. A digital image is a NumPy array
2. Indexing, slicing, and cropping
3. Arithmetic on images: dtypes, ranges, and clipping
4. Multi-channel images and plotting grids

## 🧫 The biological question

Every analysis in this course — detecting a nucleus, tracking a vesicle, measuring colocalization
between two fluorescent channels — ultimately comes down to doing arithmetic on grids of numbers.
Before we can detect a cell, we first have to be completely comfortable with the data structure a
cell *lives in* on a computer: a NumPy array. This module has no biology-specific punchline yet —
think of it as learning the alphabet before the next module teaches you to read microscope images
written in it. Every synthetic "image" you build here is a toy stand-in for the real fluorescence
images you will simulate starting in Module 02.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 1
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

No shared toolkit functions are needed yet — this module builds everything from plain NumPy and Matplotlib so you see every step.

---
## 1 · A digital image is a NumPy array

A grayscale digital image of height $H$ and width $W$ is represented as a 2-D array of shape
`(H, W)`: one number per pixel, where the number encodes how bright that pixel is. A color or
multi-channel fluorescence image adds a third axis for channel, giving shape `(H, W, C)` (or
sometimes `(C, H, W)` — you will see both conventions; always check a library's documentation).

*In words:* "row 10, column 50 of channel 0" is nothing more exotic than `image[10, 50, 0]`.

In [ ]:
# ── 1.1 build a tiny 8x8 single-channel "image" by hand ──
tiny = np.zeros((8, 8))
tiny[2:5, 2:5] = 1.0      # a small bright square, like an out-of-focus glowing blob
tiny[6, 6] = 0.6          # a single bright pixel, like a tiny particle

print('shape:', tiny.shape, '| dtype:', tiny.dtype)
print(tiny)

fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(tiny, cmap='gray', vmin=0, vmax=1)
ax.set_title('An 8x8 toy image'); ax.set_xticks(range(8)); ax.set_yticks(range(8))
plt.tight_layout(); plt.show()

🔍 **What just happened?** `tiny` is an 8×8 grid of floating-point numbers between 0 and 1.
`imshow` just maps each number to a shade of gray (0 = black, 1 = white) and draws a little colored
square for each array entry. There is no difference, conceptually, between this toy array and a
real microscope image — only the size (a real image might be 2048×2048) and how the numbers got
there (a camera sensor instead of us typing values by hand).

In [ ]:
# ── 1.2 dtype and value range matter ──
img_uint8 = (tiny * 255).astype(np.uint8)    # the convention many cameras/files use: 0-255 integers
img_float = tiny.astype(np.float64)          # the convention most analysis code prefers: 0.0-1.0 (or unbounded) floats

print('uint8  version — dtype:', img_uint8.dtype, '| min/max:', img_uint8.min(), img_uint8.max())
print('float  version — dtype:', img_float.dtype, '| min/max:', img_float.min(), img_float.max())

# A common bug: doing arithmetic on uint8 images without converting to float first.
oops = img_uint8 + img_uint8  # 128 + 128 "wraps around" in 8-bit unsigned arithmetic!
print('naive uint8 addition of the brightest pixel (should be 255, clipped):', oops.max(),
      '  <- this is WRONG if it overflowed; uint8 wraps past 255 back to 0')

### 📘 Lecture notes — dtype and value range

**Definition.** A NumPy array's `dtype` controls both how much memory each pixel uses and what
arithmetic is legal. `uint8` (unsigned 8-bit integer) can only hold whole numbers from 0 to 255 and
**silently wraps around** on overflow — no warning, no error, just a wrong answer. `float32`/`float64`
can represent fractional values and a much larger range, which is why nearly all image-processing
math in this course is done in floating point, converting back to an integer type only when saving
or displaying.

**Worked example.** `np.uint8(250) + np.uint8(10)` does not equal 260 (which cannot be represented in
8 bits) — it wraps around to `4`. This single fact is responsible for a large fraction of real-world
image-processing bugs.

**One memorable point.** *Convert to float before you do arithmetic on an image; convert back to
an integer type only at the very last step, if at all.*

✏️ **Your turn.** Predict, before running it, what `np.uint8(200) + np.uint8(100)` will print.
Then run it in a new cell and see if you were right. Now do the same computation after converting
both operands to `float`, and compare.

---
## 2 · Indexing, slicing, and cropping

In [ ]:
# ── 2.1 indexing and slicing follow the same [row, column] convention everywhere in this course ──
image = rng.uniform(0, 1, size=(10, 12))  # a random 10x12 "image"

single_pixel = image[3, 5]          # row 3, column 5
one_row      = image[3, :]          # the whole 4th row (index 3)
one_column   = image[:, 5]          # the whole 6th column (index 5)
crop         = image[2:6, 4:9]      # rows 2-5, columns 4-8: a cropped sub-image

print('single pixel value:', round(single_pixel, 3))
print('one row shape:', one_row.shape, '| one column shape:', one_column.shape)
print('crop shape:', crop.shape)

🔍 **What just happened?** NumPy slicing `a:b` means "from index `a` up to, but not including,
index `b`" — so `image[2:6, 4:9]` is 4 rows (2,3,4,5) and 5 columns (4,5,6,7,8), giving shape
`(4, 5)`. This is exactly how, later, you will crop out a small region around a detected cell or
particle for closer analysis (for example, the sub-pixel localization step in Module 13).

In [ ]:
# ── 2.2 cropping around a point of interest, the way you would around a detected cell ──
center_y, center_x = 5, 6
half_box = 3
crop_around_point = image[center_y - half_box:center_y + half_box + 1,
                           center_x - half_box:center_x + half_box + 1]
print('crop around point shape (should be 7x7):', crop_around_point.shape)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
axes[0].imshow(image, cmap='viridis'); axes[0].set_title('Full image')
axes[0].add_patch(plt.Rectangle((center_x - half_box - 0.5, center_y - half_box - 0.5),
                                 2 * half_box + 1, 2 * half_box + 1, fill=False, edgecolor='red', linewidth=2))
axes[1].imshow(crop_around_point, cmap='viridis'); axes[1].set_title('Cropped region')
plt.tight_layout(); plt.show()

✏️ **Your turn.** Change `center_y`, `center_x`, and `half_box` to crop a different region.
What happens to `crop_around_point.shape` if your box goes outside the bounds of `image`
(e.g. `center_y = 0`)? Try it, read the error or odd shape you get, and explain in a sentence why
it happened.

---
## 3 · Arithmetic on images: dtypes, ranges, and clipping

In [ ]:
# ── 3.1 brightening, scaling, and the need to clip ──
brightened = image + 0.3          # shift every pixel up — but some may now exceed 1.0!
scaled     = image * 1.5          # scale every pixel — same risk
clipped    = np.clip(brightened, 0, 1)   # force everything back into the valid [0, 1] range

print('brightened min/max (can exceed 1.0!):', round(brightened.min(), 2), round(brightened.max(), 2))
print('clipped    min/max (always in [0,1]):', round(clipped.min(), 2), round(clipped.max(), 2))

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, (title, arr) in zip(axes, [('original', image), ('brightened (unclipped)', brightened), ('brightened (clipped)', clipped)]):
    ax.imshow(arr, cmap='gray', vmin=0, vmax=1); ax.set_title(title, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

🔍 **What just happened?** `brightened` can have values above 1.0 even though we are displaying
it on a 0-to-1 grayscale — Matplotlib silently clips values outside `vmin`/`vmax` for *display*
purposes, but the underlying array still holds the out-of-range numbers until you explicitly call
`np.clip`. This distinction — "what the array actually contains" versus "how it happens to be
displayed" — causes real bugs if you forget it (e.g., computing a mean or threshold on unclipped
data gives the wrong answer even though the picture on screen looked fine).

---
## 4 · Multi-channel images and plotting grids

In [ ]:
# ── 4.1 a toy two-channel "image" (think: two fluorescence channels, e.g. a red and a green protein) ──
h, w = 10, 12
channel_red   = rng.uniform(0, 0.3, size=(h, w))
channel_green = rng.uniform(0, 0.3, size=(h, w))
channel_red[3:6, 3:6] += 0.6     # a bright patch in the red channel
channel_green[3:6, 7:10] += 0.6  # a bright patch in the green channel, a different location

two_channel = np.stack([channel_red, channel_green], axis=-1)  # shape (h, w, 2)
print('two_channel shape:', two_channel.shape, '(H, W, C) — channel axis last, as is common in Python imaging code')

rgb_composite = np.zeros((h, w, 3))
rgb_composite[..., 0] = channel_red     # R
rgb_composite[..., 1] = channel_green   # G
# blue channel left at 0

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(channel_red, cmap='Reds', vmin=0, vmax=1); axes[0].set_title('channel: red protein')
axes[1].imshow(channel_green, cmap='Greens', vmin=0, vmax=1); axes[1].set_title('channel: green protein')
axes[2].imshow(np.clip(rgb_composite, 0, 1)); axes[2].set_title('RGB composite')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

🔍 **What just happened?** We built a toy two-channel image and combined the channels into a
single RGB picture for display — exactly the operation a microscope's acquisition software performs
when you look at a "merged" multi-channel fluorescence image, except there it is done with real
intensity data from real detectors. Note that the two channels occupy *different spatial
locations* here on purpose: that non-overlap (or overlap) is precisely the kind of question
colocalization analysis (Module 26) answers quantitatively, instead of by eye.

✏️ **Your turn.** Make the two bright patches overlap instead of sitting side by side (for
example, both centered around rows 3–6, columns 3–6). Re-run the RGB composite — what new color
appears where the two channels overlap, and why (think about how red + green combine in an RGB
image)?

### 📘 Lecture notes — images as arrays, formally

**Definition.** A single-channel (grayscale) digital image is a function $I: \{0, \dots, H-1\}
\times \{0, \dots, W-1\} \rightarrow \mathbb{R}$ (or a bounded integer range), represented in code as
a 2-D array of shape `(H, W)`. A multi-channel image adds a channel axis, giving shape `(H, W, C)`.

**Worked example.** Accessing "the intensity at row $y$, column $x$, channel $c$" is `I[y, x, c]` —
note that by the strong convention in this course (and in NumPy/scikit-image generally), the *row*
index corresponds to the *vertical* ($y$) position and comes first, which can feel backwards if you
are used to thinking of $(x, y)$ coordinates first.

**One memorable point.** *Row-major, "(row, column)" indexing is the convention nearly all of
scientific Python uses for images — internalizing "row first, column second" now will save you
countless indexing bugs later in the course.*

## 📌 Key takeaways

- A digital image is nothing more than a NumPy array: 2-D for grayscale, 3-D (with a channel axis) for multi-channel.
- Always do arithmetic in floating point; convert to an integer type only at the very end, and always `np.clip` after arithmetic that could push values out of range.
- NumPy's `[row, column]` (or `[row, column, channel]`) indexing convention, with slices `a:b` meaning *up to but not including* `b`, is used everywhere in this course.
- A seeded `rng = np.random.default_rng(SEED)` makes every "random" result in this course exactly reproducible — the same seed always produces the same numbers.

## ✅ Checkpoint

**1.** What shape would a 512×512 fluorescence image with 3 channels have, as a NumPy array?

<details><summary>Answer</summary>
`(512, 512, 3)` — height, width, channels (assuming the channel-last convention used throughout
this course).
</details>

**2.** Why is it dangerous to do arithmetic directly on a `uint8` image without converting to float
first?

<details><summary>Answer</summary>
`uint8` arithmetic silently wraps around past 255 (and below 0) instead of raising an error,
producing numerically wrong results with no warning.
</details>

**3.** `image[2:5, 3:7]` — how many rows and how many columns does this slice contain?

<details><summary>Answer</summary>
3 rows (indices 2, 3, 4) and 4 columns (indices 3, 4, 5, 6) — NumPy slices exclude the upper bound.
</details>

## 🧠 Questions & discussion

- Why might a library store images as `(C, H, W)` instead of `(H, W, C)`? (Hint: think about how
  deep learning frameworks batch many images together.)
- If `rng = np.random.default_rng(0)` always produces the same sequence of "random" numbers, in
  what sense are they random at all? Why is this useful for science?
- What could go wrong if two collaborators on the same project use different, un-seeded random
  number generators when simulating test data?

## 🚀 Bonus — going further

- Build a small synthetic "checkerboard" image using array broadcasting (no loops!) and display it.
- Look up `np.einsum` and see if you can use it to compute a per-channel mean of `two_channel`
  without writing an explicit loop.
- Explore `skimage.data` as a source of built-in demonstration images *in your own exploration
  outside this notebook* (this course never downloads data inside its own notebooks, to keep the
  repository slim and self-contained, but `scikit-image` ships a few small bundled examples useful
  for your own practice).

## 📝 Lab note

```text
Module: 01 — Python & NumPy for Bioimages
Date:
One indexing mistake I made, and how I fixed it:
One thing that finally "clicked":
```

## 📚 References

1. Harris, C.R. et al. (2020). Array programming with NumPy. *Nature*, 585, 357–362.
2. Hunter, J.D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering*,
   9(3), 90–95.
3. Van der Walt, S., Colbert, S.C., Varoquaux, G. (2011). The NumPy array: a structure for efficient
   numerical computation. *Computing in Science & Engineering*, 13(2), 22–30.
4. Van der Walt, S. et al. (2014). scikit-image: image processing in Python. *PeerJ*, 2, e453.

## ➡️ Next

Module 02 puts this array fluency to work on real image-formation physics: how a microscope turns
light from a biological sample into the grid of numbers you now know how to manipulate.